In [9]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/llm-assignment/2b-assignment-RAG").resolve()
os.chdir(PROJECT_DIR)

assert (PROJECT_DIR / "domain_corpus").is_dir()
assert (PROJECT_DIR / "adapters" / "mistral7b-aws-qlora" / "adapter_config.json").is_file()
print("Project files found at:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project files found at: /content/drive/MyDrive/llm-assignment/2b-assignment-RAG


Group 162 Members (Contribution is 100% by each member)
1. 2025ae05368 - Sanjay S Betageri
2. 2025ae05921 - WUDALI LAKSHMI NARSU DHANISTA
3. 2025ae05786 - Simran Ahuja
4. 2025ae05885 - Sumit Sharma

# Assignment 2B: RAG Pipeline

## Overview
Build and evaluate a compact retrieval-augmented generation pipeline over the AWS documentation corpus prepared for Assignment 1B. The work proceeds from dense retrieval and chunk-size selection (Part A), through BM25 and hybrid retrieval (Part B), reranking (Part C), and grounded answer generation plus LLM judging (Part D). The notebook is the single executable report; saved intermediate data and results live under this folder.

| Part | Deliverable | Marks |
|---|---|---:|
| A | Dense search, chunk-size comparison, and selected chunk size | 5 |
| B | Dense, BM25, and RRF hybrid comparison | 4 |
| C | Cross-encoder reranking and ranking comparison | 4 |
| D | Grounded answers, LLM-judge results, spot-check, and failure analysis | 7 |

## Reuse from Assignment 1B
- The six cleaned AWS text files are copied into this project's `domain_corpus/` folder; no external corpus path is needed at runtime.
- The ten selected C3 prompts and the local evaluation manifest are in `data/c3_prompts.json` and `data/evaluation_queries.json`. The notebook verifies that they match.
- Assignment 1B's original evaluation file contains five prompts, while C3 provides ten prompts without gold answers. Curate a concise expected answer and exact corpus answer span for each C3 prompt before measuring hit rates; do not treat generated model outputs as gold references.
- The Assignment 1B generator is `mistralai/Mistral-7B-v0.1`; its QLoRA adapter is copied into `adapters/mistral7b-aws-qlora/`. Test grounding behavior before generating all answers because earlier outputs showed repetition and unsupported recommendations.
- The corpus, prompts, adapter, dependencies, and results are kept in this folder. The original Assignment 1B project remains unchanged.

### Requirements Setup

In [10]:
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
import sys

colab_requirements = [
    "transformers==4.46.0",
    "peft==0.12.0",
    "bitsandbytes>=0.46.0",
    "accelerate==0.34.2",
    "huggingface-hub==0.26.2",
    "sentencepiece>=0.2.0",
    "sentence-transformers==3.3.1",
    "faiss-cpu==1.10.0",
    "rank-bm25==0.2.2",
]

missing = []
for spec in colab_requirements:
    requirement = Requirement(spec)
    try:
        installed_version = version(requirement.name)
    except PackageNotFoundError:
        missing.append(spec)
        continue
    if requirement.specifier and not requirement.specifier.contains(installed_version, prereleases=True):
        missing.append(spec)

if missing:
    print("Installing or upgrading Colab packages using precompiled wheels:", missing)
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython:
        ipython.run_line_magic("pip", f"install --upgrade --only-binary=:all: {' '.join(missing)}")
    else:
        import subprocess
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", "--only-binary=:all:", *missing],
            check=True,
        )
    if any(Requirement(spec).name == "bitsandbytes" for spec in missing):
        print("Restart the Colab runtime now so it unloads the old bitsandbytes shared library, then rerun the notebook from the top.")
else:
    print("All required Colab packages are already satisfied.")

All required Colab packages are already satisfied.


## Part A: Dense Search and Chunk Size

### A0. Evaluation setup
Use the same ten C3 prompts throughout Parts A-C. Before calculating hit rates, fill `expected_answer` and `answer_span` in `data/evaluation_queries.json` for every query. `answer_span` must be a short, exact passage from one of the cleaned corpus files; a retrieved chunk is a hit when it contains that passage (ignoring case and repeated whitespace). Keep the expected answer grounded in the same source passage.

### A1. Dense retrieval
Use `BAAI/bge-small-en-v1.5` with its own tokenizer and a FAISS `IndexFlatIP` index. Embeddings and query vectors are L2-normalized, so inner product is cosine similarity. The notebook records the model's actual maximum sequence length, vector dimension, embedding batch size, and combined embedding/index build time. Chunk tokenization uses the embedding model tokenizer.

### A2. Chunk-size comparison
Compare fixed, non-overlapping 100-token and 400-token chunks. The assignment checklist specifies 100 and 400; one extracted table says 500, so this notebook follows the checklist. The selected embedding model must accommodate 400 tokens plus special tokens; the code checks this rather than silently accepting truncation. For each size, report corpus chunk count, hit rate@3 over the ten queries, and average number of chunk tokens returned in the top three. Select the Part B-D chunk size only after reviewing those results and the generator's context budget.

### Part A execution order
1. Verify the input paths and load the ten C3 prompts and six cleaned corpus files.
2. Curate the ten expected answers and answer spans in the query manifest.
3. Create token-exact chunks at 100 and 400 tokens, build dense indexes, and evaluate top-three hits.
4. Record the measured settings and choose a chunk size for Parts B-D.

In [11]:
import json
from pathlib import Path

PROJECT_DIR = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / 'data' / 'evaluation_queries.json').is_file()
     and (path / 'data' / 'c3_prompts.json').is_file()
     and (path / 'domain_corpus').is_dir()
     and (path / 'adapters' / 'mistral7b-aws-qlora' / 'adapter_config.json').is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        'Could not locate the standalone RAG project (data/, domain_corpus/, adapters/).'
    )

A2B_DIR = PROJECT_DIR
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
CORPUS_DIR = PROJECT_DIR / 'domain_corpus'
ADAPTER_PATH = PROJECT_DIR / 'adapters' / 'mistral7b-aws-qlora'
C3_PROMPTS_PATH = DATA_DIR / 'c3_prompts.json'
QUERY_MANIFEST_PATH = DATA_DIR / 'evaluation_queries.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

queries = json.loads(QUERY_MANIFEST_PATH.read_text(encoding='utf-8'))
c3_prompts = json.loads(C3_PROMPTS_PATH.read_text(encoding='utf-8'))
corpus_paths = sorted(CORPUS_DIR.glob('*.txt'))

assert len(queries) == 10, f'Expected 10 queries, found {len(queries)}.'
assert len(c3_prompts) == 10, f'Expected 10 C3 prompts, found {len(c3_prompts)}.'
assert [(q['id'], q['prompt']) for q in queries] == [(q['id'], q['prompt']) for q in c3_prompts], 'Local query manifest differs from the copied C3 prompt set.'
assert len(corpus_paths) == 6, f'Expected six copied corpus files, found {len(corpus_paths)}.'

print(f'RAG project root: {PROJECT_DIR}')
print(f'Queries: {len(queries)} (local C3 prompt set verified)')
print(f'Cleaned corpus documents: {len(corpus_paths)}')
print(f'Generator adapter: {ADAPTER_PATH}')
print('Reference answers/spans curated:', sum(bool(q.get('expected_answer') and q.get('answer_span')) for q in queries), '/ 10')

RAG project root: /content/drive/MyDrive/llm-assignment/2b-assignment-RAG
Queries: 10 (local C3 prompt set verified)
Cleaned corpus documents: 6
Generator adapter: /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/adapters/mistral7b-aws-qlora
Reference answers/spans curated: 10 / 10


### A0.1 Curate gold references
The prompt set is fixed, but C3 did not save expected answers or answer spans. Fill the two null fields per query in `data/evaluation_queries.json` by reading the corpus. Keep the span short enough to identify the relevant fact and exact enough to test whether a retrieved chunk contains it. This manual reference step is required for defensible hit-rate calculations.

In [12]:
from collections import Counter

corpus_documents = {path.stem: path.read_text(encoding='utf-8') for path in corpus_paths}

def tokenize_chunks(text, tokenizer, chunk_size, source, first_chunk_id=0):
    """Split text into fixed token windows while retaining exact source substrings."""
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True, truncation=False)
    offsets = encoded['offset_mapping']
    chunks = []
    for chunk_number, token_start in enumerate(range(0, len(offsets), chunk_size), start=first_chunk_id):
        token_end = min(token_start + chunk_size, len(offsets))
        char_start = offsets[token_start][0]
        char_end = offsets[token_end - 1][1]
        chunk_text = text[char_start:char_end].strip()
        if chunk_text:
            chunks.append({
                'id': f'{source}:{chunk_number}',
                'source': source,
                'text': chunk_text,
                'token_count': token_end - token_start,
            })
    return chunks

def build_chunks(documents, tokenizer, chunk_size):
    chunks = []
    for source, text in documents.items():
        chunks.extend(tokenize_chunks(text, tokenizer, chunk_size, source))
    return chunks

def normalize_text(text):
    return ' '.join(text.lower().split())

def answer_span_is_present(answer_span, chunk_text):
    return normalize_text(answer_span) in normalize_text(chunk_text)

### A1. Build dense retrieval
The complete Assignment 2B dependency set is in `requirements.txt`; it covers dense embeddings, FAISS, BM25, cross-encoder reranking, Mistral adapter inference, and HTML export. Install it in a separate Prayogshala environment/kernel so package changes do not affect Assignment 1B. The manifest pins PyTorch 2.3.1; confirm that this matches the lab image's CUDA runtime and use the lab-approved PyTorch wheel if needed. Do not install this stack into the Assignment 1B environment. Embeddings are normalized and searched with inner product; this is equivalent to cosine similarity for normalized vectors. The 400-token chunk size is checked against the loaded model's actual limit, including special tokens.

In [13]:
import time
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_ID = 'BAAI/bge-small-en-v1.5'
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=EMBEDDING_DEVICE)
embedding_tokenizer = embedder.tokenizer
embedding_max_length = int(embedder.max_seq_length)
special_token_count = embedding_tokenizer.num_special_tokens_to_add(pair=False)

assert 400 + special_token_count <= embedding_max_length, (
    f'400-token chunks plus {special_token_count} special tokens exceed the model limit '
    f'of {embedding_max_length}; choose a longer-context model or revise the comparison sizes.'
)

def validate_chunks_fit_model(chunks, model=embedder, batch_size=EMBEDDING_BATCH_SIZE):
    """Reject chunks that the model tokenizer would truncate, including special tokens."""
    max_length = int(model.max_seq_length)
    longest_input_length = 0
    for start in range(0, len(chunks), batch_size):
        chunk_batch = chunks[start:start + batch_size]
        encoded = model.tokenizer(
            [chunk['text'] for chunk in chunk_batch],
            add_special_tokens=True,
            padding=False,
            truncation=False,
        )
        for chunk, input_ids in zip(chunk_batch, encoded['input_ids']):
            input_length = len(input_ids)
            longest_input_length = max(longest_input_length, input_length)
            if input_length > max_length:
                raise ValueError(
                    f"Chunk {chunk['id']} encodes to {input_length} tokens including "
                    f"special tokens, exceeding the model limit of {max_length}."
                )
    return longest_input_length

def build_dense_index(chunks, model=embedder, batch_size=EMBEDDING_BATCH_SIZE):
    """Embed chunks, build a normalized-vector FAISS IndexFlatIP, and report build time."""
    if not chunks:
        raise ValueError('Cannot build an index with no chunks.')
    longest_input_length = validate_chunks_fit_model(chunks, model, batch_size)
    started = time.perf_counter()
    vectors = model.encode(
        [chunk['text'] for chunk in chunks],
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype('float32')
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    elapsed = time.perf_counter() - started
    return index, vectors.shape[1], elapsed, longest_input_length

def search_dense(query, index, chunks, model=embedder, top_k=5):
    query_vector = model.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    ).astype('float32')
    scores, indices = index.search(query_vector, min(top_k, len(chunks)))
    return [(chunks[int(i)], float(score)) for i, score in zip(indices[0], scores[0]) if i >= 0]

### A2. Evaluate 100-token and 400-token chunks
The evaluation cell builds a separate dense index for each size. Hit rate@3 is computed only when all ten answer spans are present; otherwise it stops with a clear message instead of reporting a misleading metric. Tokens sent per query is the sum of the stored token counts for the top three retrieved chunks, averaged over queries.

In [14]:
def evaluate_chunk_set(chunks, index, query_rows, top_k=3):
    if not all(row.get('answer_span') for row in query_rows):
        raise ValueError('Curate an answer_span for all ten queries before computing hit rate.')
    hits = []
    tokens_per_query = []
    for row in query_rows:
        retrieved = search_dense(row['prompt'], index, chunks, top_k=top_k)
        hits.append(any(answer_span_is_present(row['answer_span'], chunk['text']) for chunk, _ in retrieved))
        tokens_per_query.append(sum(chunk['token_count'] for chunk, _ in retrieved))
    return {
        'total_chunks': len(chunks),
        'hit_rate_at_3_pct': 100 * sum(hits) / len(hits),
        'avg_top3_tokens_per_query': sum(tokens_per_query) / len(tokens_per_query),
    }

gold_references_ready = all(row.get('expected_answer') and row.get('answer_span') for row in queries)
if not gold_references_ready:
    print('Part A retrieval functions are ready, but metrics are blocked until all ten expected answers and answer spans are curated in data/evaluation_queries.json.')
else:
    part_a_results = {}
    for chunk_size in (100, 400):
        chunks = build_chunks(corpus_documents, embedding_tokenizer, chunk_size)
        index, vector_size, build_seconds, max_chunk_input_tokens = build_dense_index(chunks)
        metrics = evaluate_chunk_set(chunks, index, queries)
        part_a_results[chunk_size] = {
            **metrics,
            'embedding_model': EMBEDDING_MODEL_ID,
            'vector_size': vector_size,
            'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)',
            'batch_size': EMBEDDING_BATCH_SIZE,
            'embedding_max_length': embedding_max_length,
            'max_chunk_input_tokens_including_special_tokens': max_chunk_input_tokens,
            'index_build_seconds': build_seconds,
        }
        print(f'{chunk_size}-token chunks: {part_a_results[chunk_size]}')
    (RESULTS_DIR / 'part_a_chunk_comparison.json').write_text(
        json.dumps(part_a_results, indent=2), encoding='utf-8'
    )
    print(f"Saved Part A metrics to {RESULTS_DIR / 'part_a_chunk_comparison.json'}")

Token indices sequence length is longer than the specified maximum sequence length for this model (29920 > 512). Running this sequence through the model will result in indexing errors


Batches:   0%|          | 0/58 [00:00<?, ?it/s]

100-token chunks: {'total_chunks': 1837, 'hit_rate_at_3_pct': 0.0, 'avg_top3_tokens_per_query': 300.0, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'vector_size': 384, 'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)', 'batch_size': 32, 'embedding_max_length': 512, 'max_chunk_input_tokens_including_special_tokens': 103, 'index_build_seconds': 3.3504586210001435}


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

400-token chunks: {'total_chunks': 461, 'hit_rate_at_3_pct': 20.0, 'avg_top3_tokens_per_query': 1200.0, 'embedding_model': 'BAAI/bge-small-en-v1.5', 'vector_size': 384, 'similarity': 'cosine similarity (L2-normalized vectors, IndexFlatIP)', 'batch_size': 32, 'embedding_max_length': 512, 'max_chunk_input_tokens_including_special_tokens': 403, 'index_build_seconds': 4.120402742000351}
Saved Part A metrics to /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/results/part_a_chunk_comparison.json


In [ ]:
from IPython.display import Markdown, display

if "part_a_results" in globals():
    rows = [
        "| Chunk size | Total chunks | Hit rate@3 (%) | Avg. top-3 tokens/query |",
        "|---:|---:|---:|---:|",
    ]
    for size in (100, 400):
        result = part_a_results[size]
        rows.append(
            f"| {size} | {result['total_chunks']} | "
            f"{result['hit_rate_at_3_pct']:.0f} | "
            f"{result['avg_top3_tokens_per_query']:.0f} |"
        )
    display(Markdown("\n".join(rows)))

from IPython.display import Markdown, display

if "part_a_results" not in globals():
    print("Run the Part A evaluation cell first to generate the decision record.")
else:
    sizes = (100, 400)
    best_size = max(
        sizes,
        key=lambda size: (
            part_a_results[size]["hit_rate_at_3_pct"],
            -part_a_results[size]["avg_top3_tokens_per_query"],
        ),
    )
    other_size = next(size for size in sizes if size != best_size)

    rows = [
        "| Chunk size | Total chunks | Hit rate@3 (%) | Avg. top-3 tokens/query |",
        "|---:|---:|---:|---:|",
    ]
    for size in sizes:
        result = part_a_results[size]
        rows.append(
            f"| {size} | {result['total_chunks']} | "
            f"{result['hit_rate_at_3_pct']:.0f} | "
            f"{result['avg_top3_tokens_per_query']:.0f} |"
        )

    query_count = len(queries)
    selected = part_a_results[best_size]
    comparison = part_a_results[other_size]
    selected_hits = round(selected["hit_rate_at_3_pct"] * query_count / 100)
    comparison_hits = round(comparison["hit_rate_at_3_pct"] * query_count / 100)

    rationale = (
        f"**Selected size and rationale:** {best_size} tokens. It achieved "
        f"{selected_hits}/{query_count} top-three hits "
        f"({selected['hit_rate_at_3_pct']:.0f}%), compared with "
        f"{comparison_hits}/{query_count} ({comparison['hit_rate_at_3_pct']:.0f}%) "
        f"for {other_size}-token chunks. The selected size averages "
        f"{selected['avg_top3_tokens_per_query']:.0f} top-three tokens per query; "
        "the choice prioritizes hit rate and breaks ties by using fewer tokens."
    )

    display(Markdown(
        "### Part A decision record\n\n"
        + "\n".join(rows)
        + "\n\n"
        + rationale
    ))

## Part B: Keyword and Hybrid Search

Use the selected 400-token chunks from Part A and the same ten query prompts and answer spans. Dense retrieval and BM25 each return their top five for comparison; RRF fuses each method's top 20, with ranks starting at 1 and `k=60`, then returns five. Average query latency excludes one warm-up run per method. Hit rates are based on the exact curated answer spans.

In [ ]:
import re
from IPython.display import Markdown, display
from rank_bm25 import BM25Okapi


SELECTED_CHUNK_SIZE = max(
    (100, 400),
    key=lambda size: (
        part_a_results[size]["hit_rate_at_3_pct"],
        -part_a_results[size]["avg_top3_tokens_per_query"],
    ),
)
chunks = build_chunks(corpus_documents, embedding_tokenizer, SELECTED_CHUNK_SIZE)
index, vector_size, build_seconds, max_chunk_input_tokens = build_dense_index(chunks)
print(f"Selected {SELECTED_CHUNK_SIZE}-token chunks using Part A hit@3; ties favor fewer top-three tokens.")


assert len(queries) == 10, f'Part B requires the same ten queries; found {len(queries)}.'
assert len(chunks) == part_a_results[SELECTED_CHUNK_SIZE]['total_chunks'], (
    'Part B chunks do not match the selected Part A chunk set.'
)
assert all(row.get('answer_span') for row in queries), 'All ten answer spans must be curated.'


def keyword_tokens(text):
    """Keep AWS identifiers such as s3:GetObject and ARN segments searchable."""
    return re.findall(r'[a-z0-9]+(?:[._:/-][a-z0-9]+)*', text.lower())


bm25_corpus = [keyword_tokens(chunk['text']) for chunk in chunks]
bm25_index = BM25Okapi(bm25_corpus)


def search_bm25(query, index=bm25_index, chunk_set=chunks, top_k=5):
    scores = index.get_scores(keyword_tokens(query))
    positions = np.argsort(-scores, kind='stable')[:min(top_k, len(chunk_set))]
    return [(chunk_set[int(position)], float(scores[position])) for position in positions]


def reciprocal_rank_fusion(dense_results, keyword_results, top_k=5, rrf_constant=60):
    """Fuse the top 20 from each retriever; a missing rank contributes no score."""
    fused_scores = {}
    chunks_by_id = {}
    for ranked_results in (dense_results[:20], keyword_results[:20]):
        for rank, (chunk, _) in enumerate(ranked_results, start=1):
            chunk_id = chunk['id']
            chunks_by_id[chunk_id] = chunk
            fused_scores[chunk_id] = fused_scores.get(chunk_id, 0.0) + 1 / (rrf_constant + rank)
    ranked_ids = sorted(fused_scores, key=lambda chunk_id: (-fused_scores[chunk_id], chunk_id))
    return [(chunks_by_id[chunk_id], fused_scores[chunk_id]) for chunk_id in ranked_ids[:top_k]]


def search_hybrid(query, top_k=5):
    dense_top20 = search_dense(query, index, chunks, top_k=20)
    bm25_top20 = search_bm25(query, top_k=20)
    return reciprocal_rank_fusion(dense_top20, bm25_top20, top_k=top_k)


def benchmark_retriever(search_function, result_count):
    """Run one untimed warm-up, then time one top-k result set per evaluation query."""
    search_function(queries[0]['prompt'], result_count)
    measured = []
    for query_row in queries:
        started = time.perf_counter()
        results = search_function(query_row['prompt'], result_count)
        measured.append({
            'results': results,
            'latency_ms': (time.perf_counter() - started) * 1000,
        })
    return measured


dense_runs = benchmark_retriever(
    lambda query, top_k: search_dense(query, index, chunks, top_k=top_k), 5
)
bm25_runs = benchmark_retriever(
    lambda query, top_k: search_bm25(query, top_k=top_k), 5
)
hybrid_runs = benchmark_retriever(search_hybrid, 5)
retrieval_runs = {
    'Dense': dense_runs,
    'BM25': bm25_runs,
    'Hybrid (RRF)': hybrid_runs,
}


def retrieval_hits(ranked_results, answer_span):
    top_three = ranked_results[:3]
    return {
        'hit_at_1': bool(ranked_results) and answer_span_is_present(answer_span, ranked_results[0][0]['text']),
        'hit_at_3': any(answer_span_is_present(answer_span, chunk['text']) for chunk, _ in top_three),
    }


part_b_results = {}
per_query_results = {}
for method_name, runs in retrieval_runs.items():
    query_records = []
    for query_row, run in zip(queries, runs):
        hits = retrieval_hits(run['results'], query_row['answer_span'])
        query_records.append({
            'query_id': query_row['id'],
            'latency_ms': run['latency_ms'],
            'hit_at_1': hits['hit_at_1'],
            'hit_at_3': hits['hit_at_3'],
            'top5_chunk_ids': [chunk['id'] for chunk, _ in run['results']],
        })
    per_query_results[method_name] = query_records
    part_b_results[method_name] = {
        'average_query_latency_ms': sum(record['latency_ms'] for record in query_records) / len(query_records),
        'hit_rate_at_1_pct': 100 * sum(record['hit_at_1'] for record in query_records) / len(query_records),
        'hit_rate_at_3_pct': 100 * sum(record['hit_at_3'] for record in query_records) / len(query_records),
        'per_query': query_records,
    }

part_b_results['settings'] = {
    'chunk_size_tokens': SELECTED_CHUNK_SIZE,
    'queries': len(queries),
    'top_k_reported': 5,
    'rrf_fusion_depth_per_retriever': 20,
    'rrf_constant': 60,
    'warmup_runs_excluded_per_method': 1,
    'bm25_tokenizer': 'lowercase regex preserving AWS identifier punctuation',
}

method_names = list(retrieval_runs)
best_method = max(
    method_names,
    key=lambda name: (
        part_b_results[name]['hit_rate_at_3_pct'],
        part_b_results[name]['hit_rate_at_1_pct'],
        -part_b_results[name]['average_query_latency_ms'],
    ),
)
per_query_by_method = {
    name: {row['query_id']: row for row in rows}
    for name, rows in per_query_results.items()
}
example_id = next(
    (row['query_id'] for row in per_query_results[best_method]
     if row['hit_at_3'] and any(
         not per_query_by_method[other][row['query_id']]['hit_at_3']
         for other in method_names if other != best_method
     )),
    queries[0]['id'],
)
example_best = per_query_by_method[best_method][example_id]
example_lagging = [
    other for other in method_names
    if not per_query_by_method[other][example_id]['hit_at_3']
]
if example_best['hit_at_3'] and example_lagging:
    example_sentence = f"For {example_id}, {best_method} retrieved the answer span in its top three while {', '.join(example_lagging)} did not."
else:
    example_sentence = f"For {example_id}, {best_method} had a top-three hit rate of {part_b_results[best_method]['hit_rate_at_3_pct']:.0f}% overall."

markdown_table = [
    '| Method | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |',
    '|---|---:|---:|---:|',
]
for method_name in method_names:
    metrics = part_b_results[method_name]
    markdown_table.append(
        f"| {method_name} | {metrics['average_query_latency_ms']:.2f} | "
        f"{metrics['hit_rate_at_1_pct']:.0f} | {metrics['hit_rate_at_3_pct']:.0f} |"
    )
summary = (
    f"**Result:** {best_method} performed best by hit rate@3 "
    f"({part_b_results[best_method]['hit_rate_at_3_pct']:.0f}%), with hit rate@1 "
    f"{part_b_results[best_method]['hit_rate_at_1_pct']:.0f}% and mean latency "
    f"{part_b_results[best_method]['average_query_latency_ms']:.2f} ms. {example_sentence}"
)
display(Markdown('\n'.join(markdown_table) + '\n\n' + summary))

(PART_B_RESULTS_PATH := RESULTS_DIR / 'part_b_retrieval_comparison.json').write_text(
    json.dumps(part_b_results, indent=2), encoding='utf-8'
)
print(f'Saved Part B metrics and per-query retrieval evidence to {PART_B_RESULTS_PATH}')

| Method | Average query latency (ms) | Hit rate@1 (%) | Hit rate@3 (%) |
|---|---:|---:|---:|
| Dense | 11.14 | 0 | 20 |
| BM25 | 2.86 | 50 | 80 |
| Hybrid (RRF) | 14.21 | 10 | 40 |

**Result:** BM25 performed best by hit rate@3 (80%), with hit rate@1 50% and mean latency 2.86 ms. For c3_01_ec2_ssh, BM25 retrieved the answer span in its top three while Dense did not.

Saved Part B metrics and per-query retrieval evidence to /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/results/part_b_retrieval_comparison.json


## Part C: Cross-Encoder Reranking

Part B selected BM25 as the current best method. This section chooses the best method from the measured Part B results, retrieves ten candidates per query, scores query-candidate pairs with `cross-encoder/ms-marco-MiniLM-L-6-v2`, and retains the top three.

### What the code does
1. Select the Part B method with the strongest hit@3, breaking ties by hit@1 and then latency.
2. Retrieve ten chunks for each of the same ten questions.
3. Score each question together with each candidate passage using a cross-encoder. The code checks pair token lengths before scoring so the reranker does not silently truncate candidates.
4. Display the top three chunk IDs and short passage previews before and after reranking for every query.
5. Report mean cross-encoder scoring time per query (after one warm-up), how often the first result changed, and hit@1 before and after. Hit@1 before uses the Part B measurement; hit@1 after checks the curated exact answer spans.

The next code cell downloads/loads the cross-encoder model if it is not cached. Review that cell before running it.

In [16]:
import time
import numpy as np
import torch
from IPython.display import Markdown, display
from sentence_transformers import CrossEncoder

RERANKER_MODEL_ID = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANKER_MAX_LENGTH = 512
RERANKER_BATCH_SIZE = 32
CANDIDATE_COUNT = 10
FINAL_RESULT_COUNT = 3

available_methods = ('Dense', 'BM25', 'Hybrid (RRF)')
best_part_b_method = max(
    (name for name in available_methods if name in part_b_results),
    key=lambda name: (
        part_b_results[name]['hit_rate_at_3_pct'],
        part_b_results[name]['hit_rate_at_1_pct'],
        -part_b_results[name]['average_query_latency_ms'],
    ),
)
assert len(queries) == 10, f'Part C requires the same ten queries; found {len(queries)}.'
assert all(row.get('answer_span') for row in queries), 'All ten answer spans must be curated.'

candidate_searchers = {
    'Dense': lambda query, top_k: search_dense(query, index, chunks, top_k=top_k),
    'BM25': lambda query, top_k: search_bm25(query, top_k=top_k),
    'Hybrid (RRF)': lambda query, top_k: search_hybrid(query, top_k=top_k),
}
candidate_sets = {
    row['id']: candidate_searchers[best_part_b_method](row['prompt'], CANDIDATE_COUNT)
    for row in queries
}
assert all(len(candidates) == CANDIDATE_COUNT for candidates in candidate_sets.values()), (
    f'Expected {CANDIDATE_COUNT} candidates for every query.'
)

reranker_device = 'cuda' if torch.cuda.is_available() else 'cpu'
reranker = CrossEncoder(
    RERANKER_MODEL_ID,
    device=reranker_device,
    max_length=RERANKER_MAX_LENGTH,
)

for query_row in queries:
    candidates = candidate_sets[query_row['id']]
    encoded_pairs = reranker.tokenizer(
        [query_row['prompt']] * len(candidates),
        [chunk['text'] for chunk, _ in candidates],
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    longest_pair_length = max(len(input_ids) for input_ids in encoded_pairs['input_ids'])
    if longest_pair_length > RERANKER_MAX_LENGTH:
        raise ValueError(
            f"{query_row['id']} has a query-passage pair of {longest_pair_length} tokens, "
            f'exceeding the reranker limit of {RERANKER_MAX_LENGTH}; no pair will be truncated.'
        )

warmup_query = queries[0]
warmup_pairs = [
    (warmup_query['prompt'], chunk['text'])
    for chunk, _ in candidate_sets[warmup_query['id']]
]
reranker.predict(
    warmup_pairs,
    batch_size=RERANKER_BATCH_SIZE,
    show_progress_bar=False,
)

part_c_per_query = []
for query_row in queries:
    candidates = candidate_sets[query_row['id']]
    pairs = [(query_row['prompt'], chunk['text']) for chunk, _ in candidates]

    rerank_started = time.perf_counter()
    scores = np.asarray(
        reranker.predict(
            pairs,
            batch_size=RERANKER_BATCH_SIZE,
            show_progress_bar=False,
        ),
        dtype='float32',
    ).reshape(-1)
    rerank_latency_ms = (time.perf_counter() - rerank_started) * 1000

    reranked_positions = np.argsort(-scores, kind='stable')[:FINAL_RESULT_COUNT]
    before_top3 = candidates[:FINAL_RESULT_COUNT]
    after_top3 = [
        (candidates[int(position)][0], float(scores[int(position)]))
        for position in reranked_positions
    ]
    before_first_id = before_top3[0][0]['id']
    after_first_id = after_top3[0][0]['id']
    before_part_b_row = next(
        row for row in per_query_results[best_part_b_method]
        if row['query_id'] == query_row['id']
    )
    after_hit_at_1 = answer_span_is_present(query_row['answer_span'], after_top3[0][0]['text'])

    def describe_ranked_chunks(ranked_chunks):
        descriptions = []
        for rank, (chunk, score) in enumerate(ranked_chunks, start=1):
            preview = ' '.join(chunk['text'].split())[:140].replace('|', '/')
            descriptions.append({
                'rank': rank,
                'chunk_id': chunk['id'],
                'source': chunk['source'],
                'score': float(score),
                'preview': preview,
            })
        return descriptions

    part_c_per_query.append({
        'query_id': query_row['id'],
        'reranking_latency_ms': rerank_latency_ms,
        'top1_changed': before_first_id != after_first_id,
        'hit_at_1_before': bool(before_part_b_row['hit_at_1']),
        'hit_at_1_after': bool(after_hit_at_1),
        'top3_before': describe_ranked_chunks(before_top3),
        'top3_after': describe_ranked_chunks(after_top3),
    })

reranking_time_per_query_ms = sum(
    row['reranking_latency_ms'] for row in part_c_per_query
) / len(part_c_per_query)
rank_change_rate_pct = 100 * sum(
    row['top1_changed'] for row in part_c_per_query
) / len(part_c_per_query)
hit_rate_at_1_before_pct = part_b_results[best_part_b_method]['hit_rate_at_1_pct']
hit_rate_at_1_after_pct = 100 * sum(
    row['hit_at_1_after'] for row in part_c_per_query
) / len(part_c_per_query)
hit_rate_change_pct = hit_rate_at_1_after_pct - hit_rate_at_1_before_pct

if hit_rate_change_pct > 0:
    reranking_conclusion = (
        f"Reranking helped on this query set: hit@1 increased from "
        f"{hit_rate_at_1_before_pct:.0f}% to {hit_rate_at_1_after_pct:.0f}% "
        f"({hit_rate_change_pct:+.0f} percentage points), at an average reranking cost "
        f"of {reranking_time_per_query_ms:.2f} ms per query."
    )
elif hit_rate_change_pct < 0:
    reranking_conclusion = (
        f"Reranking hurt hit@1 on this query set: it changed from "
        f"{hit_rate_at_1_before_pct:.0f}% to {hit_rate_at_1_after_pct:.0f}% "
        f"({hit_rate_change_pct:+.0f} percentage points), while adding "
        f"{reranking_time_per_query_ms:.2f} ms per query."
    )
else:
    reranking_conclusion = (
        f"Reranking did not change hit@1 on this query set ({hit_rate_at_1_before_pct:.0f}% "
        f"before and after), while adding {reranking_time_per_query_ms:.2f} ms per query."
    )

part_c_results = {
    'reranker_model': RERANKER_MODEL_ID,
    'reranker_max_length': RERANKER_MAX_LENGTH,
    'selected_part_b_method': best_part_b_method,
    'candidate_count': CANDIDATE_COUNT,
    'final_result_count': FINAL_RESULT_COUNT,
    'warmup_runs_excluded': 1,
    'reranking_time_per_query_ms': reranking_time_per_query_ms,
    'rank_change_rate_pct': rank_change_rate_pct,
    'hit_rate_at_1_before_pct': hit_rate_at_1_before_pct,
    'hit_rate_at_1_after_pct': hit_rate_at_1_after_pct,
    'per_query': part_c_per_query,
}

measurements_table = [
    '| Measure | Value |',
    '|---|---:|',
    f'| Reranking time per query (ms) | {reranking_time_per_query_ms:.2f} |',
    f'| Rank change rate (%) | {rank_change_rate_pct:.0f} |',
    f'| Hit rate@1 before reranking (%) | {hit_rate_at_1_before_pct:.0f} |',
    f'| Hit rate@1 after reranking (%) | {hit_rate_at_1_after_pct:.0f} |',
]
ranking_table = [
    '| Query | Top 3 before reranking | Top 3 after reranking | Top-1 changed | Hit@1 after |',
    '|---|---|---|---:|---:|',
]
for row in part_c_per_query:
    before = '<br>'.join(
        f"{item['rank']}. `{item['chunk_id']}` {item['preview']}"
        for item in row['top3_before']
    )
    after = '<br>'.join(
        f"{item['rank']}. `{item['chunk_id']}` {item['preview']}"
        for item in row['top3_after']
    )
    ranking_table.append(
        f"| {row['query_id']} | {before} | {after} | "
        f"{'Yes' if row['top1_changed'] else 'No'} | {'Yes' if row['hit_at_1_after'] else 'No'} |"
    )

display(Markdown('\n'.join(measurements_table) + '\n\n' + reranking_conclusion))
display(Markdown('\n'.join(ranking_table)))

PART_C_RESULTS_PATH = RESULTS_DIR / 'part_c_reranking_comparison.json'
PART_C_RESULTS_PATH.write_text(json.dumps(part_c_results, indent=2), encoding='utf-8')
print(f'Saved Part C rankings and metrics to {PART_C_RESULTS_PATH}')

| Measure | Value |
|---|---:|
| Reranking time per query (ms) | 49.70 |
| Rank change rate (%) | 90 |
| Hit rate@1 before reranking (%) | 50 |
| Hit rate@1 after reranking (%) | 0 |

Reranking hurt hit@1 on this query set: it changed from 50% to 0% (-50 percentage points), while adding 49.70 ms per query.

| Query | Top 3 before reranking | Top 3 after reranking | Top-1 changed | Hit@1 after |
|---|---|---|---:|---:|
| c3_01_ec2_ssh | 1. `ec2-ug:8` unsafe for production environments. In production, be sure to authorize access only from the appropriate individual IP address or range of a<br>2. `cfn-ug:51` Properties": { "BucketName": " amzn-s3-demo-bucket " } } } } YAML Resources: MyBucket: Type: AWS::S3::Bucket Properties: BucketName: amzn-s3<br>3. `ec2-ug:9` the operating system of the instance. If you can't connect to your instance, see Troubleshoot issues connecting to your Amazon EC2 Linux ins | 1. `cfn-ug:52` !Ref InstanceSecurityGroup - sg-12a4c434 KeyName: MyKey ImageId: ami-1234567890abcdef0 InstanceSecurityGroup: Type: AWS::EC2::SecurityGroup <br>2. `cfn-ug:51` Properties": { "BucketName": " amzn-s3-demo-bucket " } } } } YAML Resources: MyBucket: Type: AWS::S3::Bucket Properties: BucketName: amzn-s3<br>3. `ec2-ug:7` 't cover all possible options. To launch an instance 1. Open the Amazon EC2 console at https://console.aws.amazon.com/ec2/. 2. In the naviga | Yes | No |
| c3_02_ec2_console | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>3. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:13` an available IP address in the subnet range for you. For more information about network interfaces, see Elastic Network Interfaces in the Am<br>3. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou | No | No |
| c3_03_vpc_route | 1. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>2. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con<br>3. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou | 1. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con<br>2. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>3. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default  | Yes | No |
| c3_04_s3_getobject | 1. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i<br>2. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>3. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h | 1. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h<br>2. `iam-ug:27` permissions to work with those services aren't included in the policy. Policies and IAM groups You can organize IAM users into IAM groups an<br>3. `iam-ug:57` about that entity. •aws iam get-service-last-accessed-details-with-entities c. Learn more about the identity-based policies that an identity | Yes | No |
| c3_05_s3_public_access | 1. `s3-userguide:58` the user can create a bucket. But the user does not own the bucket; the AWS account that the user belongs to owns the bucket. The user needs<br>2. `s3-userguide:25` bucket-owner-full-control canned ACL, you can add a bucket policy that allows only object uploads that use this ACL. •Object writer – The AW<br>3. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i | 1. `s3-userguide:25` bucket-owner-full-control canned ACL, you can add a bucket policy that allows only object uploads that use this ACL. •Object writer – The AW<br>2. `s3-userguide:58` the user can create a bucket. But the user does not own the bucket; the AWS account that the user belongs to owns the bucket. The user needs<br>3. `s3-userguide:11` ended user actions and application failures. For more information, see Retaining multiple versions of objects with S3 Versioning. Version ID | Yes | No |
| c3_06_lambda_timeout | 1. `lambda-dg:23` c01d1e2e1 REPORT RequestId: c3252230-c73d-49f6-8844-968c01d1e2e1 Duration: 933.59 ms Billed Duration: 934 ms Memory Size: 128 MB Max Memory <br>2. `lambda-dg:21` kes the function handler (the Invoke phase). Failures during the Restore phase If the Restore phase fails, Lambda emits error information in<br>3. `s3-userguide:4` Simple Notiﬁcation Service (Amazon SNS), Amazon Simple Queue Service (Amazon SQS), and AWS Lambda when a change is made to your S3 resources | 1. `lambda-dg:19` Function restarts from last checkpoint •Shutdown: Cleanup of durable state and resources Topics •Init phase •Failures during the Init phase <br>2. `lambda-dg:20` You might see gaps between your function's invocation and initialization phases. Depending on your function's runtime and memory conﬁguratio<br>3. `lambda-dg:17` Watch Logs. In addition to logging your function's output, the runtime also logs entries when function invocation starts and ends. This incl | Yes | No |
| c3_07_lambda_vpc | 1. `vpc-ug:6` ﬁgure route tables Access the internet You control how the instances that you launch into a VPC access resources outside the VPC. A default <br>2. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>3. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con | 1. `vpc-ug:69` Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resou<br>2. `vpc-ug:10` will receive traﬃc only from other servers in the VPC, create a separate subnet for these servers in each Availability Zone. •If you have se<br>3. `vpc-ug:66` traﬃc, because NAT gateways do not support IPv6. To update the route table for a public subnet 1. Open the Amazon VPC console at https://con | Yes | No |
| c3_08_secretsmanager_iam | 1. `iam-ug:16` can use identity-based policies. Resource-based policies can grant cross-account access. To make a request in a diﬀerent account, a policy i<br>2. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>3. `lambda-dg:61` choose the AWS Region closest to your geographical location. aws s3api create-bucket --bucket amzn-s3-demo-bucket --region us-east-2 \ --cre | 1. `lambda-dg:13` a trigger works, start by completing the Use an Amazon S3 trigger tutorial, or for a general overview of using triggers and instructions on <br>2. `iam-ug:57` about that entity. •aws iam get-service-last-accessed-details-with-entities c. Learn more about the identity-based policies that an identity<br>3. `lambda-dg:14` . For example, if your function performs CRUD operations on a DynamoDB table, you can add the AmazonDynamoDBFullAccess policy to your role.  | Yes | No |
| c3_09_eventbridge_putevents | 1. `lambda-dg:39` potency •Use multiple AWS accounts for managing quotas Designing an application 47 Use services instead of custom code Serverless applicatio<br>2. `lambda-dg:14` . For example, if your function performs CRUD operations on a DynamoDB table, you can add the AmazonDynamoDBFullAccess policy to your role. <br>3. `lambda-dg:12` contains a runtime , a language-speciﬁc environment that relays event information and responses between Lambda and your function. Lambda pro | 1. `iam-ug:61` could look like the following example policy. JSON { "Version":"2012-10-17", "Statement": [ { "Sid": "FullAccessToSomeServices", "Effect": "<br>2. `iam-ug:0` User Guide AWS Identity and Access Management Copyright © 2026 Amazon Web Services, Inc. and/or its aﬃliates. All rights reserved. Require h<br>3. `iam-ug:54` the members of the user group and when they last accessed the service. •User – View the list of services that the user can access. You can a | Yes | No |
| c3_10_iam_role | 1. `iam-ug:38` in IAM Identity Center (either federated or created in the IAM Identity Center directory) use IAM roles to access resources. Before you star<br>2. `iam-ug:7` a speciﬁc type of resource-based policy for IAM roles. A role can have only one trust policy. •Identity-based policies (inline and managed) <br>3. `iam-ug:6` CLI to cryptographically sign requests. For more information, see AWS Signature Version 4 for API requests in the IAM User Guide . When you  | 1. `iam-ug:66` trust policies, role tags for ABAC, and conﬁgurable role paths. You can use it alongside permission sets or on its own. For more information<br>2. `iam-ug:7` a speciﬁc type of resource-based policy for IAM roles. A role can have only one trust policy. •Identity-based policies (inline and managed) <br>3. `iam-ug:21` create separate IAM users or users in IAM Identity Center for them. Instead, you can federate those user identities into AWS using either IA | Yes | No |

Saved Part C rankings and metrics to /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/results/part_c_reranking_comparison.json


## Part D: Grounded Answer Generation

This step connects the top-three chunks retained by Part C to the local Mistral-7B-v0.1 QLoRA adapter. Each passage is numbered so the model can cite evidence as `[1]`, `[2]`, or `[3]`; it must answer `Not found in the documents` when the supplied passages do not support an answer. The code uses 4-bit loading on a CUDA-enabled Colab runtime, generates one answer per evaluation query, and saves the answers and supporting chunk references to `results/part_d_generated_answers.json`.

The Hugging Face notice that `HF_TOKEN` is not set is informational for public models; authentication is optional.

In [ ]:
import json
import re
import torch
from IPython.display import Markdown, display
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError('Mistral-7B 4-bit generation requires a CUDA-enabled Colab runtime.')
assert len(part_c_per_query) == 10, 'Run Part C successfully before generating answers.'
assert ADAPTER_PATH.is_dir(), f'Local QLoRA adapter not found: {ADAPTER_PATH}'

adapter_config = json.loads((ADAPTER_PATH / 'adapter_config.json').read_text(encoding='utf-8'))
BASE_MODEL_ID = adapter_config['base_model_name_or_path']
GENERATION_MAX_NEW_TOKENS = 128
mistral_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=False)
if mistral_tokenizer.pad_token_id is None:
    mistral_tokenizer.pad_token = mistral_tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
mistral_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=False)
mistral_model.eval()
mistral_context_limit = int(mistral_model.config.max_position_embeddings)
chunks_by_id = {chunk['id']: chunk for chunk in chunks}
part_c_by_query = {row['query_id']: row for row in part_c_per_query}


def make_grounded_prompt(query_row, reranked_row):
    evidence = []
    for number, item in enumerate(reranked_row['top3_after'], start=1):
        chunk = chunks_by_id[item['chunk_id']]
        evidence.append(f"[{number}] {chunk['source']} | {chunk['id']}\n{chunk['text']}")
    evidence_text = '\n\n'.join(evidence)
    return (
        'Answer only from these passages. Cite factual claims as [1], [2], or [3]. '
        'Do not use outside knowledge. If they do not answer the question, reply exactly: '
        'Not found in the documents.\n\n'
        f"Question: {query_row['prompt']}\n\nEvidence:\n{evidence_text}\n\nAnswer:"
    )


def generate_grounded_answer(query_row, reranked_row, is_not_found_test=False):
    prompt = make_grounded_prompt(query_row, reranked_row)
    model_inputs = mistral_tokenizer(prompt, return_tensors='pt', truncation=False)
    input_token_count = int(model_inputs['input_ids'].shape[1])
    if input_token_count + GENERATION_MAX_NEW_TOKENS > mistral_context_limit:
        raise ValueError(f"{query_row['id']} exceeds the model context budget; prompt will not be truncated.")
    model_inputs = {name: tensor.to(mistral_model.device) for name, tensor in model_inputs.items()}
    with torch.inference_mode():
        generated = mistral_model.generate(
            **model_inputs,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=mistral_tokenizer.pad_token_id,
            eos_token_id=mistral_tokenizer.eos_token_id,
        )
    answer = mistral_tokenizer.decode(generated[0, input_token_count:], skip_special_tokens=True).strip()
    cited_numbers = sorted({int(number) for number in re.findall(r'\[(\d+)\]', answer)})
    return {
        'query_id': query_row['id'],
        'question': query_row['prompt'],
        'answer': answer,
        'input_token_count': input_token_count,
        'is_not_found_test': is_not_found_test,
        'cited_passage_numbers': cited_numbers,
        'citations_in_range': all(1 <= number <= len(reranked_row['top3_after']) for number in cited_numbers),
        'not_found_response_exact': (
            answer == 'Not found in the documents.' if is_not_found_test else None
        ),
        'supporting_chunks': [
            {'number': number, 'chunk_id': item['chunk_id'], 'source': item['source']}
            for number, item in enumerate(reranked_row['top3_after'], start=1)
        ],
    }

NOT_FOUND_QUERY = {
    'id': 'c3_not_found_account_id',
    'prompt': 'What is the 12-digit AWS account ID of the customer who owns the S3 bucket mentioned in these documents?',
}
not_found_candidates = candidate_searchers[best_part_b_method](NOT_FOUND_QUERY['prompt'], CANDIDATE_COUNT)
not_found_pairs = [(NOT_FOUND_QUERY['prompt'], chunk['text']) for chunk, _ in not_found_candidates]
not_found_scores = np.asarray(reranker.predict(
    not_found_pairs, batch_size=RERANKER_BATCH_SIZE, show_progress_bar=False
)).reshape(-1)
not_found_positions = np.argsort(-not_found_scores, kind='stable')[:FINAL_RESULT_COUNT]
not_found_reranked_row = {
    'top3_after': [
        {
            'chunk_id': not_found_candidates[int(position)][0]['id'],
            'source': not_found_candidates[int(position)][0]['source'],
        }
        for position in not_found_positions
    ]
}

generation_records = [
    generate_grounded_answer(query_row, part_c_by_query[query_row['id']])
    for query_row in queries
]


generation_records.append(
    generate_grounded_answer(NOT_FOUND_QUERY, not_found_reranked_row, is_not_found_test=True)
)

PART_D_GENERATION_PATH = RESULTS_DIR / 'part_d_generated_answers.json'
PART_D_GENERATION_PATH.write_text(json.dumps(generation_records, indent=2), encoding='utf-8')
average_prompt_tokens = sum(row['input_token_count'] for row in generation_records) / len(generation_records)
generation_table = [
    '| Query | Prompt tokens | Generated answer | Citations | Exact not-found test |',
    '|---|---:|---|---|---:|',
]
for record in generation_records:
    safe_answer = ' '.join(record['answer'].split()).replace('|', '/')
    exact = 'N/A' if record['not_found_response_exact'] is None else str(record['not_found_response_exact'])
    generation_table.append(
        f"| {record['query_id']} | {record['input_token_count']} | {safe_answer} | "
        f"{record['cited_passage_numbers']} | {exact} |"
    )
display(Markdown('\n'.join(generation_table)))
print(f'Average prompt size: {average_prompt_tokens:.0f} tokens; answer budget: {GENERATION_MAX_NEW_TOKENS} new tokens.')
print(f'Saved ten answers and the not-found test to {PART_D_GENERATION_PATH}')

del mistral_model, base_model
import gc
gc.collect()
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

| Query | Prompt tokens | Generated answer | Citations | Exact not-found test |
|---|---:|---|---|---:|
| c3_01_ec2_ssh | 1667 | Answer only from these passages. Cite factual claims as [1], [2], or [3]. Do not use outside knowledge. If they do not answer the question, reply exactly: Not found in the documents. Question: An EC2 Linux instance times out on SSH port 22. Give a concise diagnostic sequence covering security groups, network ACLs, routes, instance status checks, and host firewall. Evidence: [1] cfn-ug / cfn-ug:52 !Ref InstanceSecurityGroup - sg- | [1, 2, 3] | N/A |
| c3_02_ec2_console | 1569 | 1. Create a VPC with a CIDR block of 10.0.0.0/16. 2. Create a public subnet with a CIDR block of 10.0.0.0/24. 3. Create a private subnet with a CIDR block of 10.0.1.0/24. 4. Launch an instance in the public subnet with a public IPv4 address. 5. Launch an instance in the private subnet with a private IPv4 address. 6. Con | [] | N/A |
| c3_03_vpc_route | 1551 | 1. Create a VPC with a CIDR block of 10.0.0.0/16. 2. Create two public subnets in the VPC with CIDR blocks of 10.0.0.0/24 and 10.0.1.0/24. 3. Create a private subnet in the VPC with a CIDR block of 10.0.2.0/24. 4. Create an internet gateway and attach it to the VPC. 5. Update the route table | [] | N/A |
| c3_04_s3_getobject | 1497 | 1. Generate a report. The request must include the ARN of the IAM resource (user, user group, role, or policy) for which you want a report. It returns a JobId that you can then use in the GetServiceLastAccessedDetails and GetServiceLastAccessedDetailsWithEntities operations to monitor the JobStatus until the job is complete. 2. Retrieve details about the report using the JobId parameter from the previous step. Answer: 1. Generate a report. The request must include the ARN of the IAM resource (user, user group, role | [] | N/A |
| c3_05_s3_public_access | 1492 | 1. Choose Create bucket. 2. In the Bucket name ﬁeld, enter a name for your bucket. 3. (Optional) Under Region, choose the region where you want to store your bucket. 4. (Optional) Under Storage class, choose the storage class for your bucket. 5. (Optional) Under Block Public Access settings for this bucket, choose the Block Public Access settings that you want to apply to the bucket. 6. (Optional) By default, Bucket Versioning is disabled. Versioning is a means of keeping multiple variants of an object in the same bucket | [] | N/A |
| c3_06_lambda_timeout | 1561 | Lambda execution environments support both standard functions (up to 15 minutes) and Durable Functions (up to one year). While both share the same basic lifecycle, Durable Functions add state management capabilities for long-running workﬂows. Lambda invokes your function in an execution environment, which provides a secure and isolated runtime environment. The execution environment Answer: Lambda execution environments support both standard functions (up to 15 minutes) and Durable Functions (up to one year). While both share the same basic lifecycle, Durable Functions add state | [] | N/A |
| c3_07_lambda_vpc | 1573 | A Lambda function attached to a VPC times out when calling a public API. Explain the subnet and egress checks, including NAT routing and security group outbound rules. Evidence: [1] vpc-ug / vpc-ug:69 Example dual-stack VPC conﬁguration With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication between resources in your VPC and resources over the internet. The following diagram represents the architecture of your VPC. Your VPC has a | [1] | N/A |
| c3_08_secretsmanager_iam | 1460 | 1. The following are the authorization checks for identity policy, secret ARN and region, resource policy, permissions boundary, SCP, and KMS key. Identity policy: { "Version": "2012-10-17", "Statement": [ { "Effect": "Allow", "Principal": { "Service": "lambda.amazonaws.com" }, "Action": "sts:AssumeRole", "Condition": { "StringEquals": { "sts:ExternalId": "externalId" } } } ] } Secret ARN and region: | [] | N/A |
| c3_09_eventbridge_putevents | 1543 | 1. Create an IAM user. 2. Create an IAM role. 3. Create an IAM policy. 4. Attach the policy to the role. 5. Attach the role to an EC2 instance. 6. Launch the instance. 7. Verify that the instance has the role attached. 8. Verify that the instance has the policy attached. 9. Verify that the instance has the policy attached. 10. Verify that the instance has the policy attached. 11. Verify that the instance has the policy attached. 12. Verify that | [] | N/A |
| c3_10_iam_role | 1494 | 1. Create an IAM user. 2. Create an IAM role. 3. Attach a policy to the IAM role. 4. Trust the IAM role. 5. Use the IAM role. Explanation: To grant a workload access to AWS resources, you must create an IAM role and attach a policy to the role. Then, you must trust the role by adding a trust policy that grants the workload permission to assume the role. Finally, you must use the role by calling the AssumeRole API. 1. Create an IAM | [] | N/A |
| c3_not_found_account_id | 1449 | The 12-digit AWS account ID of the customer who owns the S3 bucket mentioned in these documents is 111122223333. Explanation: The 12-digit AWS account ID of the customer who owns the S3 bucket mentioned in these documents is 111122223333. The bucket name preﬁx that you create, and a suﬃx that contains your 12-digit AWS account ID, the AWS Region code, and ends with -an. | [] | False |

Average prompt size: 1532 tokens; answer budget: 128 new tokens.
Saved ten answers and the not-found test to /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/results/part_d_generated_answers.json


## Part D2: LLM Judge and Manual Review

The judge is a separate model (`Qwen/Qwen2.5-3B-Instruct`), not the Mistral answer generator. For each of the ten answerable questions, it receives the question, expected answer, generated answer, and exact three reranked passages. It returns correctness (`Correct`, `Partly correct`, or `Wrong`), whether the answer is fully supported (`Yes` or `No`), a one-sentence reason, and a quote of an unsupported claim when applicable. The out-of-corpus not-found test is excluded from these ten scored rows.

The next cell prints the per-query judge table, summary counts, three examples for manual spot-check, and one flagged answer with its three passages. After inspecting those materials, complete the manual review worksheet and decide whether the selected failure came from retrieval or generation.

In [18]:
import gc
import json
import re
import torch
from IPython.display import Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError('The 4-bit LLM judge requires a CUDA-enabled Colab runtime.')
assert len(generation_records) == 11, 'Run D1 first; expected ten answers plus the not-found test.'

for model_name in ('mistral_model', 'base_model', 'reranker', 'embedder'):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
torch.cuda.empty_cache()

JUDGE_MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
JUDGE_MAX_NEW_TOKENS = 192
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    quantization_config=judge_quantization,
    torch_dtype=torch.float16,
    device_map='auto',
)
judge_model.eval()
judge_context_limit = int(judge_model.config.max_position_embeddings)
answer_records_by_id = {
    row['query_id']: row for row in generation_records
    if not row.get('is_not_found_test', False)
}


def parse_judge_json(text, query_id):
    match = re.search(r'\{[\s\S]*\}', text)
    if match is None:
        raise ValueError(f'Judge returned no JSON object for {query_id}: {text[:500]}')
    verdict = json.loads(match.group(0))
    if verdict.get('correctness') not in ('Correct', 'Partly correct', 'Wrong'):
        raise ValueError(f'Invalid correctness label for {query_id}: {verdict}')
    if verdict.get('supported') not in ('Yes', 'No'):
        raise ValueError(f'Invalid support label for {query_id}: {verdict}')
    return verdict


judge_results = []
for query_row in queries:
    generated_row = answer_records_by_id[query_row['id']]
    reranked_row = part_c_by_query[query_row['id']]
    passages = []
    for number, item in enumerate(reranked_row['top3_after'], start=1):
        chunk = chunks_by_id[item['chunk_id']]
        passages.append(f"[{number}] Source: {chunk['source']} | Chunk: {chunk['id']}\n{chunk['text']}")
    evidence_text = '\n\n'.join(passages)
    judge_prompt = (
        'Evaluate the answer using only the supplied question, reference answer, generated answer, and passages. '
        'Correctness must be exactly Correct, Partly correct, or Wrong. '
        'Supported must be Yes only if every factual claim in the generated answer is supported by the passages; otherwise No. '
        'Treat an exact Not found response as correct only when the passages do not answer the question. '
        'Return only one JSON object with keys correctness, supported, reason, unsupported_quote. '
        'Use a one-sentence reason; unsupported_quote must be an exact unsupported quote or an empty string.\n\n'
        f"Question: {query_row['prompt']}\n"
        f"Reference answer: {query_row['expected_answer']}\n"
        f"Generated answer: {generated_row['answer']}\n"
        f"Retrieved passages:\n{evidence_text}"
    )
    rendered = judge_tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': 'You are a strict, evidence-based answer evaluator. Return valid JSON only.'},
            {'role': 'user', 'content': judge_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = judge_tokenizer(rendered, return_tensors='pt', truncation=False)
    input_length = int(inputs['input_ids'].shape[1])
    if input_length + JUDGE_MAX_NEW_TOKENS > judge_context_limit:
        raise ValueError(f'Judge prompt for {query_row["id"]} exceeds the model context budget.')
    inputs = {name: tensor.to(judge_model.device) for name, tensor in inputs.items()}
    with torch.inference_mode():
        output = judge_model.generate(
            **inputs,
            max_new_tokens=JUDGE_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=judge_tokenizer.pad_token_id or judge_tokenizer.eos_token_id,
            eos_token_id=judge_tokenizer.eos_token_id,
        )
    raw_verdict = judge_tokenizer.decode(output[0, input_length:], skip_special_tokens=True).strip()
    verdict = parse_judge_json(raw_verdict, query_row['id'])
    judge_results.append({
        'query_id': query_row['id'],
        'question': query_row['prompt'],
        'expected_answer': query_row['expected_answer'],
        'generated_answer': generated_row['answer'],
        **verdict,
        'top3_chunks': [
            {'number': number, 'chunk_id': item['chunk_id'], 'source': item['source'], 'text': chunks_by_id[item['chunk_id']]['text']}
            for number, item in enumerate(reranked_row['top3_after'], start=1)
        ],
    })

correctness_counts = {
    label: sum(row['correctness'] == label for row in judge_results)
    for label in ('Correct', 'Partly correct', 'Wrong')
}
supported_count = sum(row['supported'] == 'Yes' for row in judge_results)
judge_summary = {
    'judge_model': JUDGE_MODEL_ID,
    'total_answerable_queries': len(judge_results),
    'correctness_counts': correctness_counts,
    'supported_by_chunks_count': supported_count,
}

judge_table = [
    '| Query | Correctness | Supported | Reason | Unsupported quote |',
    '|---|---|---|---|---|',
]
for row in judge_results:
    reason = row['reason'].replace('|', '/')
    quote = row['unsupported_quote'].replace('|', '/')
    judge_table.append(
        f"| {row['query_id']} | {row['correctness']} | {row['supported']} | {reason} | {quote} |"
    )
display(Markdown('\n'.join(judge_table)))
display(Markdown(
    f"**Judge summary:** Correct {correctness_counts['Correct']}/10; "
    f"partly correct {correctness_counts['Partly correct']}/10; "
    f"wrong {correctness_counts['Wrong']}/10; supported {supported_count}/10."
))

PART_D_JUDGE_PATH = RESULTS_DIR / 'part_d_judge_results.json'
PART_D_JUDGE_PATH.write_text(json.dumps({'summary': judge_summary, 'per_query': judge_results}, indent=2), encoding='utf-8')

flagged = [row for row in judge_results if row['correctness'] != 'Correct' or row['supported'] == 'No']
spotcheck_rows = (flagged[:1] + [row for row in judge_results if row not in flagged[:1]][:2]) if flagged else judge_results[:3]
for row in spotcheck_rows:
    spotcheck = [
        f"### Manual spot-check: {row['query_id']}",
        f"**Question:** {row['question']}",
        f"**Expected:** {row['expected_answer']}",
        f"**Generated:** {row['generated_answer']}",
        f"**Judge:** {row['correctness']} / supported {row['supported']} — {row['reason']}",
    ]
    for passage in row['top3_chunks']:
        spotcheck.append(f"**[{passage['number']}] {passage['source']} ({passage['chunk_id']})**\n\n{passage['text']}")
    display(Markdown('\n\n'.join(spotcheck)))

if flagged:
    failure = flagged[0]
    failure_header = (
        f"### Failure analysis candidate: {failure['query_id']}\n"
        f"Judge: {failure['correctness']}; supported: {failure['supported']}. "
        'Inspect whether the needed fact is present in the three displayed chunks. '
        'Classify the cause as retrieval (fact absent) or generation (fact present but mishandled), then propose one change.'
    )
    display(Markdown(failure_header))
else:
    print('No wrong or unsupported answer was flagged. Select the weakest answer for the required manual failure analysis.')
print(f'Saved judge results and evidence to {PART_D_JUDGE_PATH}')

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

### Manual spot-check: c3_02_ec2_console

**Question:** I can reach an EC2 instance from the same subnet but not from the internet. List the network path checks, including public IPv4 address, internet gateway route, subnet route table, and security group.

**Expected:** Confirm the instance has a public IPv4 address, its subnet's associated route table sends 0.0.0.0/0 to an attached internet gateway, and the security group allows the required inbound port from the client. Also check the network ACL and the instance's local firewall.

**Generated:** 1. Create a VPC with a CIDR block of 10.0.0.0/16.
2. Create a public subnet with a CIDR block of 10.0.0.0/24.
3. Create a private subnet with a CIDR block of 10.0.1.0/24.
4. Launch an instance in the public subnet with a public IPv4 address.
5. Launch an instance in the private subnet with a private IPv4 address.
6. Con

**Judge:** Partly correct / supported No — The generated answer contains incorrect steps and information that are not supported by the passages. Specifically, it mentions creating a VPC with subnets and launching instances, which are not relevant to the original question about reaching an EC2 instance from the internet.

**[1] vpc-ug (vpc-ug:6)**

ﬁgure route tables
Access the internet
You control how the instances that you launch into a VPC access resources outside the VPC.
A default VPC includes an internet gateway, and each default subnet is a public subnet. Each 
instance that you launch into a default subnet has a private IPv4 address and a public IPv4 address. 
These instances can communicate with the internet through the internet gateway. An internet 
gateway enables your instances to connect to the internet through the Amazon EC2 network edge.
By default, each instance that you launch into a nondefault subnet has a private IPv4 address, but 
no public IPv4 address, unless you speciﬁcally assign one at launch, or you modify the subnet's 
public IP address attribute. These instances can communicate with each other, but can't access the 
internet.
You can enable internet access for an instance launched into a nondefault subnet by attaching an 
internet gateway to its VPC (if its VPC is not a default VPC) and associating an Elastic IP address 
with the instance.
Alternatively, to allow an instance in your VPC to initiate outbound connections to the internet 
but prevent unsolicited inbound connections from the internet, you can use a network address 
translation (NAT) device. NAT maps multiple private IPv4 addresses to a single public IPv4 address. 
You can conﬁgure the NAT device with an Elastic IP address and connect it to the internet through 
an internet gateway. This makes it possible for an instance in a private subnet to connect to the 
Route tables 8

internet through the NAT device, routing traﬃc from the instance to the internet gateway and any 
responses to the instance.
If you associate an IPv6 CIDR block with your VPC and assign IPv6 addresses to your instances, 
instances can connect to the internet over IPv6 through an internet gateway. Alternatively, 
instances can initiate outbound connections

**[2] vpc-ug (vpc-ug:13)**

an available 
IP address in the subnet range for you. For more information about network interfaces, see Elastic 
Network Interfaces in the Amazon EC2 User Guide.
You can assign additional private IP addresses, known as secondary private IP addresses, to 
instances that are running in a VPC. Unlike a primary private IP address, you can reassign a 
secondary private IP address from one network interface to another. A private IP address remains 
associated with the network interface when the instance is stopped and restarted, and is released 
when the instance is terminated. For more information about primary and secondary IP addresses, 
see Multiple IP Addresses in the Amazon EC2 User Guide.
We refer to private IP addresses as the IP addresses that are within the IPv4 CIDR range of the 
VPC. Most VPC IP address ranges fall within the private (non-publicly routable) IP address ranges 
speciﬁed in RFC 1918; however, you can use publicly routable CIDR blocks for your VPC. Regardless 
of the IP address range of your VPC, we do not support direct access to the internet from your 
VPC's CIDR block, including a publicly-routable CIDR block. You must set up internet access through 
a gateway; for example, an internet gateway, virtual private gateway, a AWS Site-to-Site VPN 
connection, or Direct Connect.
We never advertise the IPv4 address range of a subnet to the internet.
Public IPv4 addresses
All subnets have an attribute that determines whether a network interface created in the subnet 
automatically receives a public IPv4 address (also referred to as a public IP address  in this topic). 
Private IPv4 addresses 16

Therefore, when you launch an instance into a subnet that has this attribute enabled, a public IP 
address is assigned to the primary network interface that's created for the instance. A public IP 
address is mapped to the primary private IP address through network address translation (NAT).

**[3] vpc-ug (vpc-ug:69)**

Example dual-stack VPC conﬁguration
With a dual-stack conﬁguration, you can use both IPv4 and IPv6 addresses for communication 
between resources in your VPC and resources over the internet.
The following diagram represents the architecture of your VPC. Your VPC has a public subnet and a 
private subnet. The VPC and subnets have both an IPv4 CIDR block and an IPv6 CIDR block. There 
is an EC2 instance in the private subnet that has both an IPv4 address and an IPv6 address. The 
instance can send outbound IPv4 traﬃc to the internet using a NAT gateway and outbound IPv6 
traﬃc to the internet using an egress-only internet gateway.
Example dual-stack VPC 70

Route table for public subnet
The following is the route table for the public subnet. The ﬁrst two entries are the local routes. The 
third entry sends all IPv4 traﬃc to the internet gateway. Note that the fourth entry is necessary 
only if you plan to launch EC2 instances with IPv6 addresses in the public subnet.
Destination Target
VPC IPv4 CIDR local
VPC IPv6 CIDR local
0.0.0.0/0 internet-gateway-id
::/0 internet-gateway-id
Route table for the private subnet
Example dual-stack VPC 71

The following is the route table for the private subnet. The ﬁrst two entries are the local routes. 
The third entry sends all IPv4 traﬃc to the NAT gateway. The last entry sends all IPv6 traﬃc to the 
egress-only internet gateway.
Destination Target
VPC IPv4 CIDR local
VPC IPv6 CIDR local
0.0.0.0/0 nat-gateway-id
::/0 egress-only-gateway-id
AWS services that support IPv6
Computers and smart

### Manual spot-check: c3_03_vpc_route

**Question:** A private subnet cannot connect to the internet to download updates. Explain how to check its route table and NAT gateway path, and name the required route destination.

**Expected:** Check the private subnet's associated route table for an IPv4 default route (0.0.0.0/0) targeting a reachable NAT gateway in a public subnet; verify that subnet's route to the internet gateway and that the NAT gateway is available. The private subnet does not need a public IP on each instance.

**Generated:** 1. Create a VPC with a CIDR block of 10.0.0.0/16.
2. Create two public subnets in the VPC with CIDR blocks of 10.0.0.0/24 and 10.0.1.0/24.
3. Create a private subnet in the VPC with a CIDR block of 10.0.2.0/24.
4. Create an internet gateway and attach it to the VPC.
5. Update the route table

**Judge:** Wrong / supported No — The generated answer does not address the question about checking the route table and NAT gateway path for a private subnet to connect to the internet. Instead, it provides information about creating a VPC and updating route tables, which is unrelated to the question.

**[1] vpc-ug (vpc-ug:66)**

traﬃc, because NAT gateways do not support IPv6.
To update the route table for a public subnet
1. Open the Amazon VPC console at https://console.aws.amazon.com/vpc/.
2. In the navigation pane, choose Subnets . Select the public subnet. On the Route table tab, 
choose the route table ID to open the details page for the route table.
3. Select the route table. On the Routes tab, choose Edit routes.
4. Choose Add route . Choose ::/0  for Destination . Choose the ID of the internet gateway for
Target.
5. Choose Save changes.
To update the route table for a private subnet
1. Open the Amazon VPC console at https://console.aws.amazon.com/vpc/.
2. In the navigation pane, choose Egress-only internet gateways. Choose Create egress only 
internet gateway. Choose your VPC from VPC, and then choose Create egress only internet 
gateway.
For more information, see Enable outbound IPv6 traﬃc using an egress-only internet gateway.
3. In the navigation pane, choose Subnets . Select the private subnet. On the Route table tab, 
choose the route table ID to open the details page for the route table.
4. Select the route table. On the Routes tab, choose Edit routes.
5. Choose Add route . Choose ::/0  for Destination . Choose the ID of the egress-only internet 
gateway for Target.
6. Choose Save changes.
Add IPv6 support for your VPC 68

Note
A route table cannot have the same destination (::/0) pointing to both an internet 
gateway and an egress-only internet gateway simultaneously. If you receive an error 
message stating "There are existing ipv6 routes with next hop as internet Gateway" when 
conﬁguring an egre

**[2] vpc-ug (vpc-ug:10)**

will receive traﬃc only from other servers in the VPC, create a 
separate subnet for these servers in each Availability Zone.
•If you have servers that will receive traﬃc only through a VPN connection to your network, 
create a separate subnet for these servers in each Availability Zone.
Select your Availability Zones 12

If your application will receive traﬃc from the internet, the VPC must have an internet gateway. 
Attaching an internet gateway to a VPC does not automatically make your instances accessible 
from the internet. In addition to attaching the internet gateway, you must update the subnet route 
table with a route to the internet gateway. You must also ensure that the instances have public IP 
addresses and an associated security group that allows traﬃc from the internet over speciﬁc ports 
and protocols required by your application.
Alternatively, register your instances with an internet-facing load balancer. The load balancer 
receives traﬃc from the clients and distributes it across the registered instances in one or more 
Availability Zones. For more information, see Elastic Load Balancing . To allow instances in a private 
subnet to access the internet (for example, to download updates) without allowing unsolicited 
inbound connections from the internet, add a public NAT gateway in each active Availability Zone 
and update the route table to send internet traﬃc to the NAT gateway. For more information, see
the section called “Access the internet from a private subnet”.
Create your VPC
After you've determined the number of VPCs and subnets that you need, what CIDR blocks to 
assign to your VPCs and subnets, and how to connect your VPC to the internet, you are ready to 
create your VPC. If you create your VPC using the AWS Management Console and include public 
subnets in your conﬁguration, we create a route table for the subnet and add the routes required 
for direct access to the internet. For more information, see the section called “Create a VPC”.
Deploy

**[3] vpc-ug (vpc-ug:6)**

ﬁgure route tables
Access the internet
You control how the instances that you launch into a VPC access resources outside the VPC.
A default VPC includes an internet gateway, and each default subnet is a public subnet. Each 
instance that you launch into a default subnet has a private IPv4 address and a public IPv4 address. 
These instances can communicate with the internet through the internet gateway. An internet 
gateway enables your instances to connect to the internet through the Amazon EC2 network edge.
By default, each instance that you launch into a nondefault subnet has a private IPv4 address, but 
no public IPv4 address, unless you speciﬁcally assign one at launch, or you modify the subnet's 
public IP address attribute. These instances can communicate with each other, but can't access the 
internet.
You can enable internet access for an instance launched into a nondefault subnet by attaching an 
internet gateway to its VPC (if its VPC is not a default VPC) and associating an Elastic IP address 
with the instance.
Alternatively, to allow an instance in your VPC to initiate outbound connections to the internet 
but prevent unsolicited inbound connections from the internet, you can use a network address 
translation (NAT) device. NAT maps multiple private IPv4 addresses to a single public IPv4 address. 
You can conﬁgure the NAT device with an Elastic IP address and connect it to the internet through 
an internet gateway. This makes it possible for an instance in a private subnet to connect to the 
Route tables 8

internet through the NAT device, routing traﬃc from the instance to the internet gateway and any 
responses to the instance.
If you associate an IPv6 CIDR block with your VPC and assign IPv6 addresses to your instances, 
instances can connect to the internet over IPv6 through an internet gateway. Alternatively, 
instances can initiate outbound connections

### Failure analysis candidate: c3_01_ec2_ssh
Judge: Partly correct; supported: No. Inspect whether the needed fact is present in the three displayed chunks. Classify the cause as retrieval (fact absent) or generation (fact present but mishandled), then propose one change.

Saved judge results and evidence to /content/drive/MyDrive/llm-assignment/2b-assignment-RAG/results/part_d_judge_results.json


## Manual Spot-Check and Failure Analysis

After running the D2 judge cell, inspect the three displayed answer/evidence examples and compare them with the judge. Fill this table manually; do not treat the automated judge as ground truth.

| Query | Agree with judge? (Yes/No) | Explanation or correction |
|---|---|---|
| First displayed spot-check | TODO | TODO |
| Second displayed spot-check | TODO | TODO |
| Third displayed spot-check | TODO | TODO |

For the flagged failure candidate displayed above, inspect whether the needed fact is present in its three passages. If present but mishandled, classify it as a generation failure; if absent, classify it as a retrieval failure. Then record one concrete improvement.

| Failure query | Cause (retrieval/generation) | Evidence for cause | Proposed fix |
|---|---|---|---|
| TODO | TODO | TODO | TODO |

Also verify the generated out-of-corpus test in `results/part_d_generated_answers.json`: its answer should be exactly `Not found in the documents.`.